# Road Accident Data Exploration

Initial exploration of Thai road accident data (2022) for Bangkok and Phuket.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
os.chdir('/home/user/hackathon')
sys.path.insert(0, '/home/user/hackathon')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

from src.data_processing import load_raw_data, parse_datetime, extract_features, filter_city

## 1. Load and Inspect Raw Data

In [ ]:
df = load_raw_data('data/raw/accident2022.csv')
print(f'Total rows: {len(df):,}')
print(f'Total columns: {len(df.columns)}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

## 2. Missing Values Analysis

In [ ]:
key_cols = ['LATITUDE', 'LONGITUDE', 'วันที่เกิดเหตุ', 'เวลา', 'จังหวัด', 'สายทาง']
print('Missing values in key columns:')
for col in key_cols:
    missing = df[col].isna().sum()
    pct = 100 * missing / len(df)
    print(f'  {col}: {missing:,} ({pct:.1f}%)')

print(f'\nOverall missing value summary:')
print(df.isnull().sum().to_string())

## 3. City Breakdown

In [ ]:
city_counts = df['จังหวัด'].value_counts()
print('Records by province:')
print(city_counts.to_string())

bkk = df[df['จังหวัด'].str.contains('กรุงเทพ', na=False)]
phuket = df[df['จังหวัด'].str.contains('ภูเก็ต', na=False)]
print(f'\nBangkok records: {len(bkk):,}')
print(f'Phuket records: {len(phuket):,}')

## 4. Date Range

In [ ]:
df = parse_datetime(df)
df = extract_features(df)
print(f'Date range: {df["datetime"].min()} to {df["datetime"].max()}')
print(f'Duration: {(df["datetime"].max() - df["datetime"].min()).days} days')

## 5. Accidents by Hour of Day

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bangkok
bkk_data = df[df['จังหวัด'].str.contains('กรุงเทพ', na=False)]
bkk_hourly = bkk_data['hour'].value_counts().sort_index()
axes[0].bar(bkk_hourly.index, bkk_hourly.values, color='#e74c3c', alpha=0.8)
axes[0].set_title('Bangkok: Accidents by Hour', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Accidents')
axes[0].set_xticks(range(0, 24))

# Phuket
pkt_data = df[df['จังหวัด'].str.contains('ภูเก็ต', na=False)]
pkt_hourly = pkt_data['hour'].value_counts().sort_index()
axes[1].bar(pkt_hourly.index, pkt_hourly.values, color='#3498db', alpha=0.8)
axes[1].set_title('Phuket: Accidents by Hour', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Accidents')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('data/processed/accidents_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('Peak hours: 7-9am (morning rush) and 4-7pm (evening rush)')

## 6. Accidents by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (data, city, color) in enumerate([
    (bkk_data, 'Bangkok', '#e74c3c'),
    (pkt_data, 'Phuket', '#3498db')
]):
    daily = data['day_name'].value_counts().reindex(day_order)
    axes[idx].bar(range(7), daily.values, color=color, alpha=0.8)
    axes[idx].set_title(f'{city}: Accidents by Day of Week', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Day of Week')
    axes[idx].set_ylabel('Number of Accidents')
    axes[idx].set_xticks(range(7))
    axes[idx].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.savefig('data/processed/accidents_by_day.png', dpi=150, bbox_inches='tight')
plt.show()
print('Weekend spike visible, especially Friday-Saturday')

## 7. Top 10 Most Dangerous Routes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for idx, (data, city, color) in enumerate([
    (bkk_data, 'Bangkok', '#e74c3c'),
    (pkt_data, 'Phuket', '#3498db')
]):
    top_routes = data['สายทาง'].value_counts().head(10)
    axes[idx].barh(range(len(top_routes)), top_routes.values, color=color, alpha=0.8)
    axes[idx].set_yticks(range(len(top_routes)))
    axes[idx].set_yticklabels(top_routes.index, fontsize=10)
    axes[idx].set_title(f'{city}: Top 10 Dangerous Routes', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Number of Accidents')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.savefig('data/processed/top_dangerous_routes.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Accident Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (data, city) in enumerate([(bkk_data, 'Bangkok'), (pkt_data, 'Phuket')]):
    severity = data['ความรุนแรง'].value_counts()
    colors = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60']
    axes[idx].pie(severity.values, labels=severity.index, colors=colors[:len(severity)],
                  autopct='%1.1f%%', startangle=90)
    axes[idx].set_title(f'{city}: Accident Severity', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('data/processed/severity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Monthly Trend

In [ ]:
monthly = df.groupby(['month', 'จังหวัด']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
months = range(1, 13)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

if 'กรุงเทพมหานคร' in monthly.columns:
    ax.plot(months, monthly['กรุงเทพมหานคร'].values, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Bangkok')
if 'ภูเก็ต' in monthly.columns:
    ax2 = ax.twinx()
    ax2.plot(months, monthly['ภูเก็ต'].values, 's-', color='#3498db', linewidth=2, markersize=8, label='Phuket')
    ax2.set_ylabel('Phuket Accidents', color='#3498db')

ax.set_xticks(months)
ax.set_xticklabels(month_names)
ax.set_xlabel('Month')
ax.set_ylabel('Bangkok Accidents', color='#e74c3c')
ax.set_title('Monthly Accident Trend (2022)', fontsize=14, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))

plt.tight_layout()
plt.savefig('data/processed/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note spikes during January (New Year) and April (Songkran)')

## 10. Summary Statistics

In [ ]:
print('=== SUMMARY ===')
print(f'Total accidents: {len(df):,}')
print(f'Bangkok: {len(bkk_data):,} ({100*len(bkk_data)/len(df):.1f}%)')
print(f'Phuket: {len(pkt_data):,} ({100*len(pkt_data)/len(df):.1f}%)')
print(f'\nTotal fatalities: {df["จำนวนผู้เสียชีวิต"].sum():,}')
print(f'Total injuries: {df["จำนวนผู้บาดเจ็บ"].sum():,}')
print(f'\nPeak hour: {df["hour"].mode().values[0]}:00')
print(f'Worst day: {df["day_name"].mode().values[0]}')
print(f'Worst month: month {df["month"].mode().values[0]}')
print(f'\nTop accident type: {df["ประเภทอุบัติเหตุ"].mode().values[0]}')
print(f'Top cause: {df["สาเหตุ"].mode().values[0]}')